# KNN and Decision Trees

**Topic:** K-Nearest Neighbors, Decision Trees and Hyperparameter Tuning

**Name:** Manthu Harshitha

**Goal:** Predict whether a student passes or fails and compare KNN with a Decision Tree using student data.

**Target:** `passed` — 0 = Fail, 1 = Pass.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Load and understand the student data

Keep `student_exam_data.csv` in the same folder as this notebook. The CSV should contain `study_hours`, `attendance_pct`, `past_score`, `practice_tests`, `sleep_hours`, and `passed`.

In [ ]:
df = pd.read_csv("student_exam_data.csv")

print("First five rows:")
display(df.head())

print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nPass/Fail counts:")
print(df["passed"].value_counts())

## Separate features and target, then split the data

In [ ]:
features = ["study_hours", "attendance_pct", "past_score", "practice_tests", "sleep_hours"]
X = df[features]
y = df["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Scale the features for KNN

KNN uses distances, so features should be on similar scales. Fit the scaler on training data only, then transform both sets. Decision Trees do not require feature scaling.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Train and evaluate the KNN model

KNN finds the nearest training examples and predicts the majority class among them. `n_neighbors=3` means it checks the three nearest examples.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)
print("KNN Accuracy:", round(accuracy_score(y_test, knn_pred), 3))
print("\nKNN Classification Report:")
print(classification_report(y_test, knn_pred, target_names=["Fail", "Pass"], zero_division=0))

## Train and evaluate the Decision Tree model

A Decision Tree learns if-else rules by splitting the data. `max_depth=3` limits the tree size and helps reduce overfitting.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

tree_pred = tree.predict(X_test)
print("Decision Tree Accuracy:", round(accuracy_score(y_test, tree_pred), 3))
print("\nDecision Tree Classification Report:")
print(classification_report(y_test, tree_pred, target_names=["Fail", "Pass"], zero_division=0))

## Compare the two models

In [ ]:
comparison = pd.DataFrame({
    "Model": ["KNN", "Decision Tree"],
    "Test Accuracy": [
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, tree_pred)
    ]
})
display(comparison)

## Display confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, name, predictions in [
    (axes[0], "KNN", knn_pred),
    (axes[1], "Decision Tree", tree_pred)
]:
    cm = confusion_matrix(y_test, predictions)
    ax.imshow(cm, interpolation="nearest")
    ax.set_title(name)
    ax.set_xlabel("Predicted result")
    ax.set_ylabel("Actual result")
    ax.set_xticks([0, 1], ["Fail", "Pass"])
    ax.set_yticks([0, 1], ["Fail", "Pass"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

## Visualize the Decision Tree

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(
    tree,
    feature_names=features,
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True
)
plt.title("Decision Tree for Student Pass/Fail Prediction")
plt.show()

## Try different hyperparameters

In [ ]:
k_values = [1, 3, 5, 7]
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    prediction = model.predict(X_test_scaled)
    print(f"KNN with k={k}: Accuracy = {accuracy_score(y_test, prediction):.3f}")

for depth in [1, 2, 3, 5, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    print(f"Decision Tree max_depth={depth}: Accuracy = {accuracy_score(y_test, prediction):.3f}")

## Common errors to avoid

1. **Not scaling KNN features:** Large-valued columns can dominate distance calculations. Use `StandardScaler`.
2. **Scaling before splitting:** This can cause data leakage. Split first, fit the scaler on training data, and transform the test data.
3. **Making a tree too deep:** A very deep tree can memorize training data. Try limiting `max_depth`.
4. **Comparing models on different splits:** Use the same train/test split for a fair comparison.
5. **Assuming one model is always best:** Compare test metrics and choose based on the problem.

## Explain the concepts in your own words

1. **What is KNN?**  
   KNN predicts a new result by finding the nearest training examples and choosing the class that appears most often among them.

2. **What is a Decision Tree?**  
   A Decision Tree makes predictions by asking a series of feature-based questions, like an if-else process.

3. **Why do we scale features for KNN but not for Decision Trees?**  
   KNN uses distances, so features with large values can dominate. Decision Trees split one feature at a time, so scaling is usually not needed.

4. **What is the K hyperparameter in KNN?**  
   K is the number of nearest data points used to make a prediction.

5. **What does `max_depth` do in a Decision Tree?**  
   It controls how many levels the tree can grow. Limiting depth can help prevent overfitting.

6. **What is Gini Impurity?**  
   Gini Impurity measures how mixed the classes are in a tree node. A value of zero means the node contains only one class.

7. **Which model would you choose when explanations are important?**  
   I would choose a small Decision Tree because its if-else rules are easier to understand.